In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# =============================================================================
# VIỆC 1: Tạo lại features_model_a.parquet (bộ đặc trưng của mô hình nền A)
# Chạy trên Kaggle, dán vào MỘT ô code. Cần thêm dữ liệu cuộc thi "Home Credit Default Risk"
# vào notebook (Add Data -> Competition Datasets).
# Logic giữ đúng như bản thử đã chạy (để A có đúng 171 đặc trưng), chỉ thêm kiểm tra.
# =============================================================================
import os
import gc
import time
import numpy as np
import pandas as pd

T0 = time.time()
QUICK_TEST = os.environ.get("HC_QUICK_TEST") == "1"   # chỉ dùng khi chạy thử trên dữ liệu giả


def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} phút] {msg}", flush=True)


def reduce_mem_usage(df, verbose=False):
    """Giảm RAM bằng cách thu nhỏ kiểu số nguyên và số thực."""
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtype
        if str(col_type) != "category" and col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024 ** 2
    if verbose:
        print(f"RAM: {start_mem:.2f}MB -> {end_mem:.2f}MB (-{100 * (start_mem - end_mem) / start_mem:.1f}%)")
    return df


# -----------------------------------------------------------------------------
# BƯỚC 1: Tìm thư mục dữ liệu gốc
# -----------------------------------------------------------------------------
DATA_DIR = None
search_roots = [os.environ.get("HC_DATA_DIR"), "/kaggle/input", "."]
for root_dir in search_roots:
    if not root_dir or not os.path.isdir(root_dir):
        continue
    for root, dirs, files in os.walk(root_dir):
        if "application_train.csv" in files:
            DATA_DIR = root
            break
    if DATA_DIR:
        break
if DATA_DIR is None:
    raise FileNotFoundError("Không thấy application_train.csv. Đã Add Data cuộc thi Home Credit Default Risk chưa?")
for need in ["bureau.csv", "bureau_balance.csv"]:
    if not os.path.exists(os.path.join(DATA_DIR, need)):
        raise FileNotFoundError(f"Thiếu {need} trong {DATA_DIR}")
log(f"Thư mục dữ liệu: {DATA_DIR}")

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# -----------------------------------------------------------------------------
# BƯỚC 2: Đặc trưng từ bureau_balance.csv (trạng thái từng tháng của khoản vay bureau)
# -----------------------------------------------------------------------------
log("[1/3] Đang xử lý bureau_balance.csv ...")
bb = pd.read_csv(
    f"{DATA_DIR}/bureau_balance.csv",
    usecols=["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"],
    dtype={"SK_ID_BUREAU": np.int32, "MONTHS_BALANCE": np.int8, "STATUS": "category"},
)

status_mapping = {"C": 0, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "X": -1}
bb["dpd_level"] = bb["STATUS"].map(status_mapping).astype(np.int8)

for s in ["0", "1", "2", "3", "4", "5", "C", "X"]:
    bb[f"STATUS_{s}"] = (bb["STATUS"] == s).astype(np.int8)

bb["is_any_dpd"] = (bb["dpd_level"] >= 1).astype(np.int8)
bb["is_bad_31plus"] = (bb["dpd_level"] >= 2).astype(np.int8)
bb["is_bad_91plus"] = (bb["dpd_level"] >= 4).astype(np.int8)

bb_agg = bb.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
    BB_MONTHS_MIN=("MONTHS_BALANCE", "min"),
    BB_DPD_MAX=("dpd_level", "max"),
    BB_ANY_DPD_COUNT=("is_any_dpd", "sum"),
    BB_BAD_31PLUS_COUNT=("is_bad_31plus", "sum"),
    BB_BAD_91PLUS_COUNT=("is_bad_91plus", "sum"),
    BB_STATUS_0_RATIO=("STATUS_0", "mean"),
    BB_STATUS_C_RATIO=("STATUS_C", "mean"),
    BB_STATUS_1_RATIO=("STATUS_1", "mean"),
    BB_STATUS_2_RATIO=("STATUS_2", "mean"),
).reset_index()

bb_agg = reduce_mem_usage(bb_agg)
del bb
gc.collect()

# -----------------------------------------------------------------------------
# BƯỚC 3: Đặc trưng từ bureau.csv (các khoản vay ở tổ chức khác)
# -----------------------------------------------------------------------------
log("[2/3] Đang xử lý bureau.csv ...")
bureau = pd.read_csv(f"{DATA_DIR}/bureau.csv")
bureau["SK_ID_CURR"] = bureau["SK_ID_CURR"].astype(np.int32)
bureau["SK_ID_BUREAU"] = bureau["SK_ID_BUREAU"].astype(np.int32)

bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
del bb_agg
gc.collect()

bureau["is_active"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(np.int8)
bureau["is_closed"] = (bureau["CREDIT_ACTIVE"] == "Closed").astype(np.int8)

bureau["debt_to_credit_ratio"] = bureau["AMT_CREDIT_SUM_DEBT"] / (bureau["AMT_CREDIT_SUM"] + 1e-5)
bureau["overdue_to_credit_ratio"] = bureau["AMT_CREDIT_SUM_OVERDUE"] / (bureau["AMT_CREDIT_SUM"] + 1e-5)

bureau_agg_dict = {
    "SK_ID_BUREAU": ["count"],
    "DAYS_CREDIT": ["min", "max", "mean"],
    "CREDIT_DAY_OVERDUE": ["max", "mean"],
    "DAYS_CREDIT_ENDDATE": ["max", "mean"],
    "AMT_CREDIT_SUM": ["sum", "max", "mean"],
    "AMT_CREDIT_SUM_DEBT": ["sum", "mean"],
    "AMT_CREDIT_SUM_OVERDUE": ["sum", "max"],
    "AMT_ANNUITY": ["sum", "max"],
    "is_active": ["sum", "mean"],
    "is_closed": ["sum"],
    "debt_to_credit_ratio": ["mean", "max"],
    "overdue_to_credit_ratio": ["mean", "max"],
    "BB_MONTHS_COUNT": ["sum", "mean"],
    "BB_MONTHS_MIN": ["min"],
    "BB_DPD_MAX": ["max"],
    "BB_ANY_DPD_COUNT": ["sum", "max"],
    "BB_BAD_31PLUS_COUNT": ["sum", "max"],
    "BB_BAD_91PLUS_COUNT": ["sum", "max"],
    "BB_STATUS_0_RATIO": ["mean"],
    "BB_STATUS_C_RATIO": ["mean"],
}

bureau_client_agg = bureau.groupby("SK_ID_CURR").agg(bureau_agg_dict)
bureau_client_agg.columns = pd.Index([f"B_{col[0]}_{col[1].upper()}" for col in bureau_client_agg.columns.tolist()])
bureau_client_agg.reset_index(inplace=True)

bureau_client_agg["B_TOTAL_DEBT_TO_TOTAL_CREDIT"] = (
    bureau_client_agg["B_AMT_CREDIT_SUM_DEBT_SUM"] / (bureau_client_agg["B_AMT_CREDIT_SUM_SUM"] + 1e-5)
)
bureau_client_agg["B_ACTIVE_LOANS_RATIO"] = (
    bureau_client_agg["B_is_active_SUM"] / (bureau_client_agg["B_SK_ID_BUREAU_COUNT"] + 1e-5)
)

bureau_client_agg = reduce_mem_usage(bureau_client_agg)
del bureau
gc.collect()

# -----------------------------------------------------------------------------
# BƯỚC 4: application_train.csv và ghép thành bộ đặc trưng A
# -----------------------------------------------------------------------------
log("[3/3] Đang xử lý application_train.csv và ghép dữ liệu ...")
app = pd.read_csv(f"{DATA_DIR}/application_train.csv")

# Các tỷ lệ đòn bẩy tài chính từ đơn vay
app["RATIO_ANNUITY_TO_INCOME"] = app["AMT_ANNUITY"] / (app["AMT_INCOME_TOTAL"] + 1e-5)
app["RATIO_CREDIT_TO_INCOME"] = app["AMT_CREDIT"] / (app["AMT_INCOME_TOTAL"] + 1e-5)
app["RATIO_CREDIT_TO_ANNUITY"] = app["AMT_CREDIT"] / (app["AMT_ANNUITY"] + 1e-5)
app["RATIO_GOODS_TO_CREDIT"] = app["AMT_GOODS_PRICE"] / (app["AMT_CREDIT"] + 1e-5)
app["RATIO_INCOME_PER_FAMILY_MEMBER"] = app["AMT_INCOME_TOTAL"] / (app["CNT_FAM_MEMBERS"] + 1e-5)

# Giá trị bất thường: Home Credit gán 365243 ngày cho người hưu trí/thất nghiệp
app["DAYS_EMPLOYED_ANOM"] = (app["DAYS_EMPLOYED"] == 365243).astype(np.int8)
app["DAYS_EMPLOYED"] = app["DAYS_EMPLOYED"].replace(365243, np.nan)
app["RATIO_EMPLOYED_TO_BIRTH"] = app["DAYS_EMPLOYED"] / (app["DAYS_BIRTH"] + 1e-5)

# Thống kê ba điểm tín dụng ngoài
ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
app["EXT_SOURCES_MEAN"] = app[ext].mean(axis=1)
app["EXT_SOURCES_STD"] = app[ext].std(axis=1)
app["EXT_SOURCES_MIN"] = app[ext].min(axis=1)
app["EXT_SOURCES_MAX"] = app[ext].max(axis=1)

features_a = app.merge(bureau_client_agg, on="SK_ID_CURR", how="left")
del app, bureau_client_agg
gc.collect()

# Cờ có lịch sử bureau hay không
features_a["HAS_BUREAU_RECORD"] = features_a["B_SK_ID_BUREAU_COUNT"].notnull().astype(np.int8)

# Tương tác chéo giữa đơn vay và bureau
features_a["CROSS_TOTAL_DEBT_TO_INCOME"] = (
    (features_a["AMT_CREDIT"] + features_a["B_AMT_CREDIT_SUM_DEBT_SUM"].fillna(0))
    / (features_a["AMT_INCOME_TOTAL"] + 1e-5)
)

# Cột chữ -> category để LightGBM nhận trực tiếp
cat_cols = [c for c in features_a.columns
            if c not in ("SK_ID_CURR", "TARGET")
            and not pd.api.types.is_numeric_dtype(features_a[c])
            and str(features_a[c].dtype) != "category"]
for c in cat_cols:
    features_a[c] = features_a[c].astype("category")

features_a = reduce_mem_usage(features_a, verbose=True)
features_a = features_a.sort_values("SK_ID_CURR").reset_index(drop=True)

# -----------------------------------------------------------------------------
# BƯỚC 5: Kiểm tra trước khi lưu
# -----------------------------------------------------------------------------
print("\n== Kiểm tra features_model_a ==")
problems = []


def check(name, ok, detail=""):
    print(("  ✓ " if ok else "  ✗ ") + name + (f" – {detail}" if detail else ""))
    if not ok:
        problems.append(name)


n_rows, n_cols = features_a.shape
n_features = n_cols - 2   # trừ SK_ID_CURR và TARGET
check("Có cột SK_ID_CURR và TARGET", {"SK_ID_CURR", "TARGET"} <= set(features_a.columns))
check("SK_ID_CURR không trùng và đã sắp tăng dần",
      features_a["SK_ID_CURR"].is_unique and bool(np.all(np.diff(features_a["SK_ID_CURR"].to_numpy()) > 0)))
check("TARGET chỉ có 0 và 1, không thiếu",
      set(features_a["TARGET"].unique()) <= {0, 1} and not features_a["TARGET"].isna().any())
rate = float(features_a["TARGET"].mean())
print(f"  Số dòng: {n_rows:,} | số cột: {n_cols} | số đặc trưng: {n_features} | tỷ lệ TARGET = {rate:.4%}")
if not QUICK_TEST:
    check("Đủ 307,511 dòng", n_rows == 307_511, f"{n_rows:,}")
    check("Tỷ lệ TARGET khoảng 8,07%", abs(rate - 0.0807) < 0.0005, f"{rate:.4%}")
    check("Đúng 171 đặc trưng như bản thử", n_features == 171, f"hiện có {n_features}")
all_nan = [c for c in features_a.columns if features_a[c].isna().all()]
check("Không có cột nào toàn thiếu", not all_nan, str(all_nan[:5]))
cov = float(features_a["HAS_BUREAU_RECORD"].mean())
print(f"  Tỷ lệ khách có bản ghi bureau: {cov:.1%}")
dup_cols = features_a.columns[features_a.columns.duplicated()].tolist()
check("Không có tên cột trùng", not dup_cols, str(dup_cols))

if problems and not QUICK_TEST:
    raise AssertionError("Các kiểm tra sau không đạt, KHÔNG lưu file: " + "; ".join(problems))

# -----------------------------------------------------------------------------
# BƯỚC 6: Lưu file
# -----------------------------------------------------------------------------
output_path = os.path.join(OUT_DIR, "features_model_a.parquet")
features_a.to_parquet(output_path, index=False)
size_mb = os.path.getsize(output_path) / 1024 ** 2
log(f"Đã lưu {output_path} ({size_mb:.0f} MB).")
print("\nCác cột đầu tiên:", list(features_a.columns[:8]))
print("Các cột cuối cùng:", list(features_a.columns[-6:]))
print("\n=> Gửi tôi: toàn bộ log trên (đặc biệt phần '== Kiểm tra ==') để sang Việc 2.")

In [ ]:
# =============================================================================
# VIỆC 2: Tạo lại features_model_b.parquet (A cộng 48 đặc trưng hành vi nội bộ)
# Chạy trên Kaggle TRONG CÙNG notebook đã chạy Việc 1 (cần file features_model_a.parquet).
# Dán vào MỘT ô code. Logic giữ đúng bản thử để B có đúng 219 đặc trưng.
# =============================================================================
import os
import gc
import time
import numpy as np
import pandas as pd

T0 = time.time()
QUICK_TEST = os.environ.get("HC_QUICK_TEST") == "1"   # chỉ dùng khi chạy thử trên dữ liệu giả


def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} phút] {msg}", flush=True)


def reduce_mem_usage(df, verbose=False):
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtype
        if str(col_type) != "category" and col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024 ** 2
    if verbose:
        print(f"RAM: {start_mem:.2f}MB -> {end_mem:.2f}MB (-{100 * (start_mem - end_mem) / start_mem:.1f}%)")
    return df


# -----------------------------------------------------------------------------
# BƯỚC 1: Tìm dữ liệu gốc và file features_model_a.parquet
# -----------------------------------------------------------------------------
def find_dir_with(filename, roots):
    for root_dir in roots:
        if not root_dir or not os.path.isdir(root_dir):
            continue
        for root, dirs, files in os.walk(root_dir):
            if filename in files:
                return root
    return None


DATA_DIR = find_dir_with("application_train.csv", [os.environ.get("HC_DATA_DIR"), "/kaggle/input", "."])
if DATA_DIR is None:
    raise FileNotFoundError("Không thấy application_train.csv. Đã Add Data cuộc thi Home Credit Default Risk chưa?")
for need in ["installments_payments.csv", "POS_CASH_balance.csv", "credit_card_balance.csv", "previous_application.csv"]:
    if not os.path.exists(os.path.join(DATA_DIR, need)):
        raise FileNotFoundError(f"Thiếu {need} trong {DATA_DIR}")

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
A_DIR = find_dir_with("features_model_a.parquet", [os.environ.get("HC_A_DIR"), OUT_DIR, "/kaggle/input", "."])
if A_DIR is None:
    raise FileNotFoundError("Không thấy features_model_a.parquet. Hãy chạy Việc 1 trong cùng notebook trước.")
features_a_path = os.path.join(A_DIR, "features_model_a.parquet")
log(f"Dữ liệu gốc: {DATA_DIR}\nFile A: {features_a_path}")

train_ids = set(pd.read_csv(f"{DATA_DIR}/application_train.csv", usecols=["SK_ID_CURR"])["SK_ID_CURR"])

# -----------------------------------------------------------------------------
# BƯỚC 2: installments_payments.csv (từng kỳ trả góp: trễ hay sớm, thiếu hay đủ)
# -----------------------------------------------------------------------------
log("[1/4] Đang xử lý installments_payments.csv ...")
ins = pd.read_csv(f"{DATA_DIR}/installments_payments.csv")
ins = ins[ins["SK_ID_CURR"].isin(train_ids)]

# DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT: > 0 là trả trễ, < 0 là trả sớm
ins["DPD"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)
ins["DBD"] = (ins["DAYS_INSTALMENT"] - ins["DAYS_ENTRY_PAYMENT"]).clip(lower=0)

ins["PAYMENT_DEFICIT"] = (ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]).clip(lower=0)
ins["PAYMENT_RATIO"] = ins["AMT_PAYMENT"] / (ins["AMT_INSTALMENT"] + 1e-5)

ins["IS_LATE"] = (ins["DPD"] > 0).astype(np.int8)
ins["IS_UNDERPAID"] = (ins["PAYMENT_DEFICIT"] > 0).astype(np.int8)

ins_agg_all = ins.groupby("SK_ID_CURR").agg(
    INS_NUM_PAYMENTS=("SK_ID_PREV", "count"),
    INS_DPD_MAX=("DPD", "max"),
    INS_DPD_MEAN=("DPD", "mean"),
    INS_DBD_MEAN=("DBD", "mean"),
    INS_LATE_RATIO=("IS_LATE", "mean"),
    INS_UNDERPAID_RATIO=("IS_UNDERPAID", "mean"),
    INS_PAYMENT_RATIO_MEAN=("PAYMENT_RATIO", "mean"),
    INS_PAYMENT_DEFICIT_SUM=("PAYMENT_DEFICIT", "sum"),
    INS_DAYS_LAST_PAYMENT=("DAYS_ENTRY_PAYMENT", "max"),
)

# Cửa sổ thời gian gần: 180 ngày (6M) và 365 ngày (12M). DAYS_ENTRY_PAYMENT là số âm.
for days, label in [(180, "6M"), (365, "12M")]:
    mask = ins["DAYS_ENTRY_PAYMENT"] >= -days
    ins_sub = ins[mask]
    sub_agg = ins_sub.groupby("SK_ID_CURR").agg(**{
        f"INS_DPD_MAX_{label}": ("DPD", "max"),
        f"INS_DPD_MEAN_{label}": ("DPD", "mean"),
        f"INS_LATE_RATIO_{label}": ("IS_LATE", "mean"),
        f"INS_UNDERPAID_RATIO_{label}": ("IS_UNDERPAID", "mean"),
    })
    ins_agg_all = ins_agg_all.join(sub_agg)

# Xu hướng: tỷ lệ trễ 6 tháng gần nhất so với toàn thời gian
ins_agg_all["INS_LATE_TREND_6M_VS_ALL"] = (
    ins_agg_all["INS_LATE_RATIO_6M"] / (ins_agg_all["INS_LATE_RATIO"] + 1e-5)
)

ins_client_agg = reduce_mem_usage(ins_agg_all.reset_index())
del ins, ins_agg_all
gc.collect()

# -----------------------------------------------------------------------------
# BƯỚC 3: POS_CASH_balance.csv
# -----------------------------------------------------------------------------
log("[2/4] Đang xử lý POS_CASH_balance.csv ...")
pos = pd.read_csv(f"{DATA_DIR}/POS_CASH_balance.csv")
pos = pos[pos["SK_ID_CURR"].isin(train_ids)]

pos["IS_OVERDUE"] = (pos["SK_DPD"] > 0).astype(np.int8)
pos["IS_COMPLETED"] = (pos["NAME_CONTRACT_STATUS"] == "Completed").astype(np.int8)
pos["IS_ACTIVE"] = (pos["NAME_CONTRACT_STATUS"] == "Active").astype(np.int8)

pos_agg = pos.groupby("SK_ID_CURR").agg(
    POS_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
    POS_MONTHS_LAST=("MONTHS_BALANCE", "max"),
    POS_DPD_MAX=("SK_DPD", "max"),
    POS_DPD_MEAN=("SK_DPD", "mean"),
    POS_OVERDUE_RATIO=("IS_OVERDUE", "mean"),
    POS_COMPLETED_SUM=("IS_COMPLETED", "sum"),
    POS_ACTIVE_SUM=("IS_ACTIVE", "sum"),
    POS_REMAINING_INSTALMENTS_MEAN=("CNT_INSTALMENT_FUTURE", "mean"),
)

pos_12m = pos[pos["MONTHS_BALANCE"] >= -12].groupby("SK_ID_CURR").agg(
    POS_DPD_MAX_12M=("SK_DPD", "max"),
    POS_OVERDUE_RATIO_12M=("IS_OVERDUE", "mean"),
)
pos_agg = pos_agg.join(pos_12m)

pos_client_agg = reduce_mem_usage(pos_agg.reset_index())
del pos, pos_agg
gc.collect()

# -----------------------------------------------------------------------------
# BƯỚC 4: credit_card_balance.csv
# -----------------------------------------------------------------------------
log("[3/4] Đang xử lý credit_card_balance.csv ...")
cc = pd.read_csv(f"{DATA_DIR}/credit_card_balance.csv")
cc = cc[cc["SK_ID_CURR"].isin(train_ids)]

cc["UTILIZATION"] = cc["AMT_BALANCE"] / (cc["AMT_CREDIT_LIMIT_ACTUAL"] + 1e-5)
cc["CASH_DRAWINGS_RATIO"] = cc["AMT_DRAWINGS_ATM_CURRENT"] / (cc["AMT_BALANCE"] + 1e-5)
cc["PAID_MIN_RATIO"] = cc["AMT_PAYMENT_CURRENT"] / (cc["AMT_INST_MIN_REGULARITY"] + 1e-5)
cc["IS_DPD"] = (cc["SK_DPD"] > 0).astype(np.int8)

cc_agg = cc.groupby("SK_ID_CURR").agg(
    CC_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
    CC_BALANCE_MEAN=("AMT_BALANCE", "mean"),
    CC_BALANCE_MAX=("AMT_BALANCE", "max"),
    CC_UTILIZATION_MEAN=("UTILIZATION", "mean"),
    CC_UTILIZATION_MAX=("UTILIZATION", "max"),
    CC_CASH_DRAWINGS_RATIO_MEAN=("CASH_DRAWINGS_RATIO", "mean"),
    CC_PAID_MIN_RATIO_MEAN=("PAID_MIN_RATIO", "mean"),
    CC_DPD_MAX=("SK_DPD", "max"),
    CC_DPD_MONTHS_SUM=("IS_DPD", "sum"),
)

cc_6m = cc[cc["MONTHS_BALANCE"] >= -6].groupby("SK_ID_CURR").agg(
    CC_UTILIZATION_MEAN_6M=("UTILIZATION", "mean"),
    CC_DPD_MAX_6M=("SK_DPD", "max"),
)
cc_agg = cc_agg.join(cc_6m)

cc_client_agg = reduce_mem_usage(cc_agg.reset_index())
del cc, cc_agg
gc.collect()

# -----------------------------------------------------------------------------
# BƯỚC 5: previous_application.csv
# -----------------------------------------------------------------------------
log("[4/4] Đang xử lý previous_application.csv ...")
prev = pd.read_csv(f"{DATA_DIR}/previous_application.csv")
prev = prev[prev["SK_ID_CURR"].isin(train_ids)]

prev["IS_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(np.int8)
prev["IS_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(np.int8)
prev["APPLICATION_CREDIT_RATIO"] = prev["AMT_CREDIT"] / (prev["AMT_APPLICATION"] + 1e-5)

prev_agg = prev.groupby("SK_ID_CURR").agg(
    PREV_APP_COUNT=("SK_ID_PREV", "count"),
    PREV_APPROVED_SUM=("IS_APPROVED", "sum"),
    PREV_REFUSED_SUM=("IS_REFUSED", "sum"),
    PREV_APPROVED_RATIO=("IS_APPROVED", "mean"),
    PREV_REFUSED_RATIO=("IS_REFUSED", "mean"),
    PREV_DAYS_DECISION_LAST=("DAYS_DECISION", "max"),
    PREV_CREDIT_APPL_RATIO_MEAN=("APPLICATION_CREDIT_RATIO", "mean"),
    PREV_DOWN_PAYMENT_RATE_MEAN=("RATE_DOWN_PAYMENT", "mean"),
)

prev_client_agg = reduce_mem_usage(prev_agg.reset_index())
del prev, prev_agg
gc.collect()

# -----------------------------------------------------------------------------
# BƯỚC 6: Ghép vào bộ đặc trưng A để thành bộ B
# -----------------------------------------------------------------------------
log("Đang ghép các bảng hành vi vào A ...")
features_a = pd.read_parquet(features_a_path)
n_cols_a = features_a.shape[1]
cols_a = list(features_a.columns)

features_b = features_a.merge(ins_client_agg, on="SK_ID_CURR", how="left")
features_b = features_b.merge(pos_client_agg, on="SK_ID_CURR", how="left")
features_b = features_b.merge(cc_client_agg, on="SK_ID_CURR", how="left")
features_b = features_b.merge(prev_client_agg, on="SK_ID_CURR", how="left")
del ins_client_agg, pos_client_agg, cc_client_agg, prev_client_agg
gc.collect()

# Cờ: có lịch sử nội bộ tại tổ chức này hay không
features_b["HAS_INTERNAL_HISTORY"] = (
    features_b["INS_NUM_PAYMENTS"].notnull()
    | features_b["POS_MONTHS_COUNT"].notnull()
    | features_b["PREV_APP_COUNT"].notnull()
).astype(np.int8)

features_b = reduce_mem_usage(features_b, verbose=True)

# -----------------------------------------------------------------------------
# BƯỚC 7: Kiểm tra trước khi lưu
# -----------------------------------------------------------------------------
EXPECTED_NEW = (
    ["INS_NUM_PAYMENTS", "INS_DPD_MAX", "INS_DPD_MEAN", "INS_DBD_MEAN", "INS_LATE_RATIO", "INS_UNDERPAID_RATIO",
     "INS_PAYMENT_RATIO_MEAN", "INS_PAYMENT_DEFICIT_SUM", "INS_DAYS_LAST_PAYMENT",
     "INS_DPD_MAX_6M", "INS_DPD_MEAN_6M", "INS_LATE_RATIO_6M", "INS_UNDERPAID_RATIO_6M",
     "INS_DPD_MAX_12M", "INS_DPD_MEAN_12M", "INS_LATE_RATIO_12M", "INS_UNDERPAID_RATIO_12M",
     "INS_LATE_TREND_6M_VS_ALL"]
    + ["POS_MONTHS_COUNT", "POS_MONTHS_LAST", "POS_DPD_MAX", "POS_DPD_MEAN", "POS_OVERDUE_RATIO",
       "POS_COMPLETED_SUM", "POS_ACTIVE_SUM", "POS_REMAINING_INSTALMENTS_MEAN",
       "POS_DPD_MAX_12M", "POS_OVERDUE_RATIO_12M"]
    + ["CC_MONTHS_COUNT", "CC_BALANCE_MEAN", "CC_BALANCE_MAX", "CC_UTILIZATION_MEAN", "CC_UTILIZATION_MAX",
       "CC_CASH_DRAWINGS_RATIO_MEAN", "CC_PAID_MIN_RATIO_MEAN", "CC_DPD_MAX", "CC_DPD_MONTHS_SUM",
       "CC_UTILIZATION_MEAN_6M", "CC_DPD_MAX_6M"]
    + ["PREV_APP_COUNT", "PREV_APPROVED_SUM", "PREV_REFUSED_SUM", "PREV_APPROVED_RATIO", "PREV_REFUSED_RATIO",
       "PREV_DAYS_DECISION_LAST", "PREV_CREDIT_APPL_RATIO_MEAN", "PREV_DOWN_PAYMENT_RATE_MEAN"]
    + ["HAS_INTERNAL_HISTORY"]
)

print("\n== Kiểm tra features_model_b ==")
problems, warnings_ = [], []


def check(name, ok, detail="", hard=True):
    print(("  ✓ " if ok else ("  ✗ " if hard else "  ⚠ ")) + name + (f" – {detail}" if detail else ""))
    if not ok:
        (problems if hard else warnings_).append(name)


n_rows, n_cols = features_b.shape
new_cols = [c for c in features_b.columns if c not in set(cols_a)]
check("Số dòng bằng bộ A", n_rows == len(features_a), f"B={n_rows:,} A={len(features_a):,}")
check("SK_ID_CURR không trùng và sắp tăng dần",
      features_b["SK_ID_CURR"].is_unique and bool(np.all(np.diff(features_b["SK_ID_CURR"].to_numpy()) > 0)))
check("SK_ID_CURR và TARGET giống hệt bộ A",
      bool(np.array_equal(features_b["SK_ID_CURR"].to_numpy(), features_a["SK_ID_CURR"].to_numpy())
           and np.array_equal(features_b["TARGET"].to_numpy(), features_a["TARGET"].to_numpy())))
check("Các cột của A giữ nguyên thứ tự ở đầu B", list(features_b.columns[:n_cols_a]) == cols_a)
check("Các cột của A có giá trị giống hệt trong B", bool(features_b[cols_a].equals(features_a)))
check("Đúng 48 cột hành vi mới, đúng tên", set(new_cols) == set(EXPECTED_NEW) and len(new_cols) == 48,
      f"thiếu: {sorted(set(EXPECTED_NEW) - set(new_cols))[:5]} thừa: {sorted(set(new_cols) - set(EXPECTED_NEW))[:5]}")
if not QUICK_TEST:
    check("Đủ 307,511 dòng", n_rows == 307_511, f"{n_rows:,}")
    check("Đủ 221 cột (219 đặc trưng cộng ID và TARGET)", n_cols == 221, f"hiện có {n_cols}")
all_nan = [c for c in new_cols if features_b[c].isna().all()]
check("Không có cột hành vi nào toàn thiếu", not all_nan, str(all_nan[:5]))
check("HAS_INTERNAL_HISTORY chỉ có 0 và 1", set(features_b["HAS_INTERNAL_HISTORY"].unique()) <= {0, 1})
n_cold = int((features_b["HAS_INTERNAL_HISTORY"] == 0).sum())
print(f"  Khách KHÔNG có lịch sử nội bộ: {n_cold:,} ({n_cold / n_rows:.2%})")
if not QUICK_TEST:
    check("Số khách không có lịch sử nội bộ = 14,758 như bản thử", n_cold == 14_758, f"hiện có {n_cold:,}", hard=False)
for prefix in ("INS_", "POS_", "CC_", "PREV_"):
    c0 = [c for c in new_cols if c.startswith(prefix)][0]
    print(f"  Tỷ lệ có dữ liệu ({c0}): {features_b[c0].notnull().mean():.1%}")

if problems and not QUICK_TEST:
    raise AssertionError("Các kiểm tra sau không đạt, KHÔNG lưu file: " + "; ".join(problems))

# -----------------------------------------------------------------------------
# BƯỚC 8: Lưu file
# -----------------------------------------------------------------------------
output_path_b = os.path.join(OUT_DIR, "features_model_b.parquet")
features_b.to_parquet(output_path_b, index=False)
log(f"Đã lưu {output_path_b} ({os.path.getsize(output_path_b) / 1024 ** 2:.0f} MB).")
print(f"Số đặc trưng A: {n_cols_a - 2} | B: {n_cols - 2} | thêm: {n_cols - n_cols_a}")
if warnings_:
    print("Cảnh báo cần xem lại:", warnings_)
print("\n=> Gửi tôi toàn bộ log trên (phần '== Kiểm tra ==') để sang Việc 3.")

In [ ]:
# =============================================================================
# Ô CODE KHÔI PHỤC TỪ FILE UPLOAD + CHẠY VIỆC 4 & VIỆC 5 + NÉN FILE ZIP
# Chỉ cần Upload 11 file đã tải về vào mục Input của Kaggle rồi chạy ô này.
# =============================================================================
import os
import gc
import json
import time
import shutil
import zipfile
import numpy as np
import pandas as pd
import lightgbm as lgb
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score, average_precision_score

T0 = time.time()
QUICK_TEST = os.environ.get("HC_QUICK_TEST") == "1"
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
EVAL_DIR = os.path.join(OUT_DIR, "eval_results")
MODEL_DIR = os.path.join(OUT_DIR, "models")
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} phút] {msg}", flush=True)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500.0, 500.0)))


def find_file(filename, roots):
    for r in roots:
        if not r or not os.path.isdir(r):
            continue
        for root, _, files in os.walk(r):
            if filename in files:
                return os.path.join(root, filename)
    return None


# -----------------------------------------------------------------------------
# BƯỚC 0: Khôi phục các file bạn đã upload từ /kaggle/input sang /kaggle/working
# -----------------------------------------------------------------------------
roots = [OUT_DIR, "/kaggle/input", "."]
saved_files = [
    "oof_predictions.csv",
    "calibration_deciles_ab.csv",
    "calibration_deciles_ab.png",
    "config_shared.json",
    "fold_seed_auc_report.csv",
    "monotone_constraints.csv",
    "feature_list_a.txt",
    "feature_list_b.txt",
    "folds.csv",
]

log("[0/5] Đang tìm và đồng bộ các file đã upload từ /kaggle/input ...")
for fn in saved_files:
    dst = os.path.join(OUT_DIR, fn)
    if not os.path.exists(dst):
        src = find_file(fn, roots)
        if src is None:
            raise FileNotFoundError(f"Không tìm thấy {fn} trong /kaggle/input. Hãy kiểm tra lại Dataset bạn vừa Upload!")
        shutil.copy2(src, dst)
        log(f"  + Đã khôi phục: {fn} từ {src}")

feat_a_path = find_file("features_model_a.parquet", roots)
feat_b_path = find_file("features_model_b.parquet", roots)
app_train_path = find_file("application_train.csv", roots)
if not feat_b_path or not app_train_path:
    raise FileNotFoundError("Cần có features_model_b.parquet và dữ liệu cuộc thi Home Credit (application_train.csv) trong Input!")

oof = pd.read_csv(os.path.join(OUT_DIR, "oof_predictions.csv")).sort_values("SK_ID_CURR").reset_index(drop=True)
folds = pd.read_csv(os.path.join(OUT_DIR, "folds.csv")).sort_values("SK_ID_CURR").reset_index(drop=True)
fb = pd.read_parquet(feat_b_path).sort_values("SK_ID_CURR").reset_index(drop=True)

with open(os.path.join(OUT_DIR, "config_shared.json"), "r", encoding="utf-8") as f:
    cfg = json.load(f)
with open(os.path.join(OUT_DIR, "feature_list_a.txt"), "r", encoding="utf-8") as f:
    feats_a = [x.strip() for x in f if x.strip() and not x.startswith("#")]
with open(os.path.join(OUT_DIR, "feature_list_b.txt"), "r", encoding="utf-8") as f:
    feats_b = [x.strip() for x in f if x.strip() and not x.startswith("#")]

mono_df = pd.read_csv(os.path.join(OUT_DIR, "monotone_constraints.csv"))
mono_map = dict(zip(mono_df["feature"], mono_df.iloc[:, 1]))

# Kiểm tra nếu chưa có model_b trong models/ thì tạo lại để tính SHAP
existing_b_models = [fn for fn in os.listdir(MODEL_DIR) if fn.startswith("model_b_fold")]
if len(existing_b_models) < 15:
    fixed_rounds = int(cfg.get("fixed_num_boost_round", 532))
    seeds_b = cfg.get("seeds_model_b", [42, 123, 2024])
    lgb_params = cfg["lgbm_params"]
    mono_vec = [
        0 if (str(fb[c].dtype) == "category" or not pd.api.types.is_numeric_dtype(fb[c]))
        else int(mono_map.get(c, 0))
        for c in feats_b
    ]
    y_arr = oof["TARGET"].to_numpy()
    fold_arr = folds["fold"].to_numpy()
    log(f"Đang dựng lại 15 file model_b ({fixed_rounds} vòng cố định) để trích xuất SHAP ...")
    for seed in seeds_b:
        params = dict(lgb_params, seed=seed, feature_fraction_seed=seed, bagging_seed=seed, monotone_constraints=mono_vec)
        for f_idx in range(5):
            m_file = os.path.join(MODEL_DIR, f"model_b_fold{f_idx}_seed{seed}.txt")
            if os.path.exists(m_file):
                continue
            tr_m = fold_arr != f_idx
            dtr = lgb.Dataset(fb.loc[tr_m, feats_b], label=y_arr[tr_m])
            bst = lgb.train(params, dtr, num_boost_round=fixed_rounds)
            bst.save_model(m_file)
            del dtr, bst
            gc.collect()
        log(f"  + Xong 5 fold model_b cho seed={seed}")

# -----------------------------------------------------------------------------
# BƯỚC 1: Tạo frozen_cohort_labels_N12.csv chuẩn bảng A2
# -----------------------------------------------------------------------------
EXPECTED_A2 = {
    "cohort_t2_N12": {"never_bad": 85865, "unrehabilitated": 3168, "rehabilitated": 3072, "excluded_missing_data": 126, "no_bureau_balance": 215280},
    "cohort_t3_N12": {"never_bad": 88826, "unrehabilitated": 1774, "rehabilitated": 1554, "excluded_missing_data": 77, "no_bureau_balance": 215280},
    "cohort_t4_N12": {"never_bad": 89620, "unrehabilitated": 1444, "rehabilitated": 1103, "excluded_missing_data": 64, "no_bureau_balance": 215280},
}

existing_cohort = find_file("frozen_cohort_labels_N12.csv", roots)
if existing_cohort:
    cohorts = pd.read_csv(existing_cohort).sort_values("SK_ID_CURR").reset_index(drop=True)
    cohorts.to_csv(os.path.join(OUT_DIR, "frozen_cohort_labels_N12.csv"), index=False)
else:
    log("[1/5] Đang tạo frozen_cohort_labels_N12.csv từ bureau.csv và bureau_balance.csv ...")
    data_dir = os.path.dirname(app_train_path)
    bb = pd.read_csv(f"{data_dir}/bureau_balance.csv", usecols=["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"])
    bur = pd.read_csv(f"{data_dir}/bureau.csv", usecols=["SK_ID_CURR", "SK_ID_BUREAU"])
    bur = bur[bur["SK_ID_CURR"].isin(set(oof["SK_ID_CURR"]))]

    s_map = {"C": 0, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "X": -1}
    bb["dpd"] = bb["STATUS"].map(s_map).fillna(-1).astype(np.int8)
    m_bb = bur.merge(bb, on="SK_ID_BUREAU", how="inner")
    del bb, bur
    gc.collect()

    cohorts = pd.DataFrame({"SK_ID_CURR": oof["SK_ID_CURR"].to_numpy()})
    best_win = -12
    for win_bound in [-11, -12]:
        m_bb["rec_known"] = ((m_bb["MONTHS_BALANCE"] >= win_bound) & (m_bb["dpd"] >= 0)).astype(np.int8)
        m_bb["bad2"] = (m_bb["dpd"] >= 2).astype(np.int8)
        m_bb["rec_bad2"] = ((m_bb["MONTHS_BALANCE"] >= win_bound) & (m_bb["dpd"] >= 2)).astype(np.int8)
        tmp_agg = m_bb.groupby("SK_ID_CURR").agg(rec_known=("rec_known", "sum"), bad2=("bad2", "sum"), rec_bad2=("rec_bad2", "sum")).reset_index()
        tmp_df = cohorts.merge(tmp_agg, on="SK_ID_CURR", how="left")
        conds = [tmp_df["bad2"].isna(), tmp_df["bad2"] == 0, tmp_df["rec_bad2"] > 0, (tmp_df["bad2"] > 0) & (tmp_df["rec_bad2"] == 0) & (tmp_df["rec_known"] > 0)]
        lbls = np.select(conds, ["no_bureau_balance", "never_bad", "unrehabilitated", "rehabilitated"], default="excluded_missing_data")
        if pd.Series(lbls).value_counts().to_dict() == EXPECTED_A2["cohort_t2_N12"]:
            best_win = win_bound
            break

    m_bb["rec_known"] = ((m_bb["MONTHS_BALANCE"] >= best_win) & (m_bb["dpd"] >= 0)).astype(np.int8)
    for t in [2, 3, 4]:
        m_bb[f"bad_t{t}"] = (m_bb["dpd"] >= t).astype(np.int8)
        m_bb[f"rec_bad_t{t}"] = ((m_bb["MONTHS_BALANCE"] >= best_win) & (m_bb["dpd"] >= t)).astype(np.int8)

    agg = m_bb.groupby("SK_ID_CURR").agg(
        rec_known=("rec_known", "sum"),
        bad_t2=("bad_t2", "sum"), rec_bad_t2=("rec_bad_t2", "sum"),
        bad_t3=("bad_t3", "sum"), rec_bad_t3=("rec_bad_t3", "sum"),
        bad_t4=("bad_t4", "sum"), rec_bad_t4=("rec_bad_t4", "sum"),
    ).reset_index()
    del m_bb
    gc.collect()

    cohorts = cohorts.merge(agg, on="SK_ID_CURR", how="left")
    for t in [2, 3, 4]:
        conds = [
            cohorts[f"bad_t{t}"].isna(),
            cohorts[f"bad_t{t}"] == 0,
            cohorts[f"rec_bad_t{t}"] > 0,
            (cohorts[f"bad_t{t}"] > 0) & (cohorts[f"rec_bad_t{t}"] == 0) & (cohorts["rec_known"] > 0),
        ]
        cohorts[f"cohort_t{t}_N12"] = np.select(conds, ["no_bureau_balance", "never_bad", "unrehabilitated", "rehabilitated"], default="excluded_missing_data")
    cohorts = cohorts[["SK_ID_CURR", "cohort_t2_N12", "cohort_t3_N12", "cohort_t4_N12"]].sort_values("SK_ID_CURR").reset_index(drop=True)
    cohorts.to_csv(os.path.join(OUT_DIR, "frozen_cohort_labels_N12.csv"), index=False)

# -----------------------------------------------------------------------------
# BƯỚC 2: Quy đổi điểm 1 - 1001, 10 hạng & tính điểm cộng/trừ SHAP
# -----------------------------------------------------------------------------
log("[2/5] Đang tính điểm 1 - 1001, chia 10 hạng và phân rã điểm SHAP ...")
p_bar = float(oof["TARGET"].mean())
anchor_logit = float(np.log(p_bar / (1.0 - p_bar)))
logit_a = oof["logit_model_a"].to_numpy()
logit_b = oof["logit_model_b"].to_numpy()

chosen_pdo = 40
for pdo in sorted([40, 50, 60, 80, 100], reverse=True):
    fac = pdo / np.log(2.0)
    raw_a = 706.0 - fac * (logit_a - anchor_logit)
    raw_b = 706.0 - fac * (logit_b - anchor_logit)
    if np.mean((raw_a <= 1.0) | (raw_a >= 1001.0)) <= 0.001 and np.mean((raw_b <= 1.0) | (raw_b >= 1001.0)) <= 0.001:
        chosen_pdo = int(pdo)
        break

factor = float(chosen_pdo / np.log(2.0))
offset = float(706.0 + factor * anchor_logit)
score_a = np.clip(706.0 - factor * (logit_a - anchor_logit), 1.0, 1001.0)
score_b = np.clip(706.0 - factor * (logit_b - anchor_logit), 1.0, 1001.0)

quantiles = np.quantile(score_b, np.linspace(0.0, 1.0, 11))
internal_cuts = quantiles[1:-1]


def assign_grade(scores, cuts):
    return (10 - np.digitize(scores, cuts, right=False)).astype(int)


grade_a = assign_grade(score_a, internal_cuts)
grade_b = assign_grade(score_b, internal_cuts)
grade_of_706 = int(assign_grade(np.array([706.0]), internal_cuts)[0])
grade_label_map = {1: "rất tốt", 2: "rất tốt", 3: "tốt", 4: "tốt", 5: "trung bình", 6: "trung bình", 7: "dưới trung bình", 8: "dưới trung bình", 9: "xấu", 10: "xấu"}

scored_customers = pd.DataFrame({
    "SK_ID_CURR": oof["SK_ID_CURR"].to_numpy(),
    "score_a": score_a,
    "score_b": score_b,
    "grade_a": grade_a,
    "grade_b": grade_b,
    "score_diff": score_b - score_a,
}).sort_values("SK_ID_CURR").reset_index(drop=True)
scored_customers.to_csv(os.path.join(OUT_DIR, "scored_customers.csv"), index=False)

grade_rows = []
for g in range(1, 11):
    m = grade_b == g
    low_b = float(quantiles[10 - g]) if g < 10 else 1.0
    high_b = float(quantiles[11 - g]) if g > 1 else 1001.0
    bad_r = float(oof.loc[m, "TARGET"].mean())
    grade_rows.append({
        "grade": int(g), "label": grade_label_map[g],
        "score_min": round(low_b, 2), "score_max": round(high_b, 2),
        "n_customers": int(m.sum()), "pct_customers": float(m.mean() * 100.0),
        "bad_rate": bad_r, "bad_per_100": round(bad_r * 100.0, 2),
    })
grade_table_df = pd.DataFrame(grade_rows)
grade_table_df.to_csv(os.path.join(EVAL_DIR, "grade_table_10_b.csv"), index=False)

expected_prop = np.array([np.mean(grade_b == g) for g in range(1, 11)])
psi_by_fold = {}
for f_idx in range(5):
    f_grades = grade_b[(oof["fold"] == f_idx).to_numpy()]
    actual_prop = np.clip(np.array([np.mean(f_grades == g) for g in range(1, 11)]), 1e-6, 1.0)
    psi_by_fold[f"fold_{f_idx}"] = float(np.sum((actual_prop - expected_prop) * np.log(actual_prop / expected_prop)))

scoring_params = {
    "pdo": chosen_pdo, "factor": factor, "offset": offset,
    "anchor_bad_rate": p_bar, "anchor_logit": anchor_logit, "anchor_score": 706.0,
    "clip_min": 1.0, "clip_max": 1001.0,
    "clip_rate_a": float(np.mean((score_a <= 1.0) | (score_a >= 1001.0))),
    "clip_rate_b": float(np.mean((score_b <= 1.0) | (score_b >= 1001.0))),
    "decile_cutoffs": [float(x) for x in quantiles],
    "internal_cutoffs": [float(x) for x in internal_cuts],
    "grade_labels": {str(k): v for k, v in grade_label_map.items()},
    "grade_of_score_706": grade_of_706,
    "psi_by_fold_b": psi_by_fold,
    "max_psi_b": float(max(psi_by_fold.values())),
    "strictly_monotonic_bad_rate_b": bool(np.all(np.diff(grade_table_df["bad_rate"].to_numpy()) > 0)),
}
with open(os.path.join(OUT_DIR, "scoring_params.json"), "w", encoding="utf-8") as f:
    json.dump(scoring_params, f, indent=2, ensure_ascii=False)


def assign_feature_group(col):
    if col.startswith("EXT_SOURCE"):
        return ("ext_source_scores", "Điểm tín dụng từ nguồn dữ liệu bên ngoài")
    if col.startswith("INS_"):
        return ("internal_installment_behavior", "Kỷ luật trả góp nội bộ (trả đúng hạn, trước hạn)")
    if col.startswith("POS_"):
        return ("internal_pos_cash_behavior", "Lịch sử vay trả góp POS và vay tiền mặt nội bộ")
    if col.startswith("CC_"):
        return ("internal_credit_card_behavior", "Mức sử dụng hạn mức và trả nợ thẻ tín dụng")
    if col.startswith("PREV_") or col == "HAS_INTERNAL_HISTORY":
        return ("internal_previous_app_history", "Lịch sử phê duyệt các đơn đăng ký vay trước đây")
    if col.startswith("B_BB_") or "OVERDUE" in col or "DPD" in col:
        return ("bureau_delinquency_history", "Lịch sử quá hạn tại các tổ chức tín dụng khác")
    if col.startswith("B_") or col in ("HAS_BUREAU_RECORD", "CROSS_TOTAL_DEBT_TO_INCOME"):
        return ("bureau_credit_exposure", "Quy mô dư nợ và khoản vay đang mở tại Bureau")
    if any(k in col for k in ("INCOME", "CREDIT", "ANNUITY", "GOODS")):
        return ("application_financial_ratios", "Tỷ lệ thu nhập, khoản vay và gánh nặng trả góp")
    if any(k in col for k in ("DAYS_", "HOUR_", "NAME_", "CODE_", "FLAG_", "CNT_", "OWN_", "OCCUPATION", "ORGANIZATION", "REG_", "LIVE_")):
        return ("applicant_demographics_stability", "Độ tuổi, thâm niên làm việc và thông tin cá nhân")
    return ("other_factors", "Các yếu tố bổ sung về nơi ở và hồ sơ")


fmap_rows = [{"feature": c, "group_code": assign_feature_group(c)[0], "group_description_vi": assign_feature_group(c)[1]} for c in feats_b]
fmap_df = pd.DataFrame(fmap_rows)
fmap_df.to_csv(os.path.join(OUT_DIR, "feature_group_mapping.csv"), index=False)

rehab_mask = (cohorts["cohort_t2_N12"] == "rehabilitated").to_numpy()
unrehab_mask = (cohorts["cohort_t2_N12"] == "unrehabilitated").to_numpy()
never_mask = (cohorts["cohort_t2_N12"] == "never_bad").to_numpy()
rng = np.random.default_rng(42)
shap_eval_indices = np.unique(np.concatenate([
    np.where(rehab_mask)[0],
    rng.choice(np.where(unrehab_mask)[0], size=min(1000, int(unrehab_mask.sum())), replace=False),
    rng.choice(np.where(never_mask)[0], size=min(2000, int(never_mask.sum())), replace=False),
]))

shap_sub_df = fb.iloc[shap_eval_indices].copy().reset_index(drop=True)
sub_folds = oof.loc[shap_eval_indices, "fold"].to_numpy()
sub_scores_b = scored_customers.loc[shap_eval_indices, "score_b"].to_numpy()
sub_scores_a = scored_customers.loc[shap_eval_indices, "score_a"].to_numpy()
sub_cohorts = cohorts.loc[shap_eval_indices, "cohort_t2_N12"].to_numpy()

shap_matrix = np.zeros((len(shap_eval_indices), len(feats_b) + 1), dtype=np.float64)
for f_idx in range(5):
    f_pos = np.where(sub_folds == f_idx)[0]
    if len(f_pos) == 0:
        continue
    x_part = shap_sub_df.loc[f_pos, feats_b]
    m_files = sorted([os.path.join(MODEL_DIR, fn) for fn in os.listdir(MODEL_DIR) if fn.startswith(f"model_b_fold{f_idx}_seed")])
    fold_c = np.zeros((len(f_pos), len(feats_b) + 1), dtype=np.float64)
    for mf in m_files:
        bst = lgb.Booster(model_file=mf)
        fold_c += bst.predict(x_part, pred_contrib=True)
    shap_matrix[f_pos] = fold_c / len(m_files)

base_scores = 706.0 - factor * (shap_matrix[:, -1] - anchor_logit)
point_matrix = -factor * shap_matrix[:, :-1]
group_codes = sorted(fmap_df["group_code"].unique())

shap_points_df = pd.DataFrame({
    "SK_ID_CURR": shap_sub_df["SK_ID_CURR"].to_numpy(),
    "cohort_t2_N12": sub_cohorts,
    "base_score": base_scores,
    "score_a": sub_scores_a,
    "score_b": sub_scores_b,
})
for g_code in group_codes:
    g_idx = [i for i in range(len(feats_b)) if fmap_rows[i]["group_code"] == g_code]
    shap_points_df[f"pts_{g_code}"] = point_matrix[:, g_idx].sum(axis=1)

shap_points_df.to_parquet(os.path.join(OUT_DIR, "shap_points_by_group.parquet"), index=False)
pts_cols = [f"pts_{g}" for g in group_codes]
shap_points_df.groupby("cohort_t2_N12")[pts_cols + ["base_score", "score_a", "score_b"]].mean().reset_index().to_csv(
    os.path.join(EVAL_DIR, "shap_group_points_by_cohort.csv"), index=False
)

rehab_up = shap_points_df[(shap_points_df["cohort_t2_N12"] == "rehabilitated") & ((shap_points_df["score_b"] - shap_points_df["score_a"]) >= 40.0)].sort_values("score_b", ascending=False)
rehab_same = shap_points_df[(shap_points_df["cohort_t2_N12"] == "rehabilitated") & ((shap_points_df["score_b"] - shap_points_df["score_a"]).abs() <= 5.0)]
unrehab_ex = shap_points_df[shap_points_df["cohort_t2_N12"] == "unrehabilitated"].sort_values("score_b", ascending=True)
never_ex = shap_points_df[shap_points_df["cohort_t2_N12"] == "never_bad"].sort_values("score_b", ascending=False)

demo_rows = []
for case_label, sub_table in [
    ("rehabilitated_upgraded", rehab_up),
    ("rehabilitated_unchanged", rehab_same),
    ("unrehabilitated_high_risk", unrehab_ex),
    ("never_bad_clean_history", never_ex),
]:
    if len(sub_table) > 0:
        r0 = sub_table.iloc[0]
        sk_id = int(r0["SK_ID_CURR"])
        oof_row = oof[oof["SK_ID_CURR"] == sk_id].iloc[0]
        sc_row = scored_customers[scored_customers["SK_ID_CURR"] == sk_id].iloc[0]
        demo_rows.append({
            "case_type": case_label,
            "sk_id_curr": sk_id,
            "target_actual": int(oof_row["TARGET"]),
            "score_a": round(float(sc_row["score_a"]), 2),
            "grade_a": int(sc_row["grade_a"]),
            "pred_risk_a_pct": round(float(oof_row["pred_risk_model_a"] * 100.0), 2),
            "score_b": round(float(sc_row["score_b"]), 2),
            "grade_b": int(sc_row["grade_b"]),
            "pred_risk_b_pct": round(float(oof_row["pred_risk_model_b"] * 100.0), 2),
            "score_diff": round(float(sc_row["score_diff"]), 2),
            "base_score_b": round(float(r0["base_score"]), 2),
            "group_points_b": {g: round(float(r0[f"pts_{g}"]), 2) for g in group_codes},
        })
with open(os.path.join(EVAL_DIR, "demo_customer_scorecards.json"), "w", encoding="utf-8") as f:
    json.dump(demo_rows, f, indent=2, ensure_ascii=False)

# -----------------------------------------------------------------------------
# BƯỚC 3: Kiểm định thống kê Bootstrap 1 000 lần (Người 4)
# -----------------------------------------------------------------------------
df = oof.merge(fb[["SK_ID_CURR", "HAS_INTERNAL_HISTORY"]], on="SK_ID_CURR", how="left").merge(cohorts, on="SK_ID_CURR", how="left")
del fb
gc.collect()

recomp_auc_a = float(roc_auc_score(df["TARGET"], df["pred_risk_model_a"]))
recomp_auc_b = float(roc_auc_score(df["TARGET"], df["pred_risk_model_b"]))
ind_check_df = pd.DataFrame([
    {"model": "model_a", "reported_auc": cfg["oof_auc_a"], "recomputed_auc": recomp_auc_a, "abs_diff": abs(recomp_auc_a - cfg["oof_auc_a"])},
    {"model": "model_b", "reported_auc": cfg["oof_auc_b"], "recomputed_auc": recomp_auc_b, "abs_diff": abs(recomp_auc_b - cfg["oof_auc_b"])},
])
ind_check_df.to_csv(os.path.join(EVAL_DIR, "independent_auc_check.csv"), index=False)

n_boot = 100 if QUICK_TEST else 1000
log(f"[3/5] Đang chạy Bootstrap ghép cặp {n_boot} lần ...")


def _one_boot_iter(n, y, pa, pb, seed_i, fallback_da, fallback_dp):
    rng = np.random.default_rng(seed_i)
    idx = rng.integers(0, n, size=n)
    yi = y[idx]
    if yi.min() == yi.max():
        return fallback_da, fallback_dp
    return (
        float(roc_auc_score(yi, pb[idx]) - roc_auc_score(yi, pa[idx])),
        float(average_precision_score(yi, pb[idx]) - average_precision_score(yi, pa[idx])),
    )


def bootstrap_group_metrics(y, pa, pb, n_boot=1000, seed=42):
    n = len(y)
    if n < 5 or len(np.unique(y)) < 2:
        return None, None, None
    auc_a0 = float(roc_auc_score(y, pa))
    auc_b0 = float(roc_auc_score(y, pb))
    pra_a0 = float(average_precision_score(y, pa))
    pra_b0 = float(average_precision_score(y, pb))
    da0, dp0 = auc_b0 - auc_a0, pra_b0 - pra_a0

    master_rng = np.random.default_rng(seed)
    iter_seeds = master_rng.integers(0, 2**31 - 1, size=n_boot)
    pairs = Parallel(n_jobs=-1, prefer="threads")(
        delayed(_one_boot_iter)(n, y, pa, pb, int(s), da0, dp0) for s in iter_seeds
    )
    d_auc = np.array([p[0] for p in pairs], dtype=np.float64)
    d_pra = np.array([p[1] for p in pairs], dtype=np.float64)
    auc_lo, auc_hi = np.percentile(d_auc, [2.5, 97.5])
    pra_lo, pra_hi = np.percentile(d_pra, [2.5, 97.5])
    return {
        "n_customers": int(n), "observed_bad_rate": float(np.mean(y)),
        "auc_a": auc_a0, "auc_b": auc_b0, "delta_auc": float(da0),
        "delta_auc_ci95_low": float(auc_lo), "delta_auc_ci95_high": float(auc_hi),
        "pr_auc_a": pra_a0, "pr_auc_b": pra_b0, "delta_pr_auc": float(dp0),
        "delta_pr_auc_ci95_low": float(pra_lo), "delta_pr_auc_ci95_high": float(pra_hi),
        "conclusion_auc": "cải thiện" if auc_lo > 0 else "chưa đủ bằng chứng",
        "conclusion_pr_auc": "cải thiện" if pra_lo > 0 else "chưa đủ bằng chứng",
    }, d_auc, d_pra


metric_rows = []
r_all, _, _ = bootstrap_group_metrics(df["TARGET"].to_numpy(), df["pred_risk_model_a"].to_numpy(), df["pred_risk_model_b"].to_numpy(), n_boot=n_boot)
metric_rows.append({"config": "ALL", "group": "full_population", **r_all})

cold_sub = df[df["HAS_INTERNAL_HISTORY"] == 0]
r_cold, _, _ = bootstrap_group_metrics(cold_sub["TARGET"].to_numpy(), cold_sub["pred_risk_model_a"].to_numpy(), cold_sub["pred_risk_model_b"].to_numpy(), n_boot=n_boot)
metric_rows.append({"config": "ALL", "group": "no_internal_history", **r_cold})

nobb_sub = df[df["cohort_t2_N12"] == "no_bureau_balance"]
r_nobb_cached, b_auc_nobb, b_pra_nobb = bootstrap_group_metrics(nobb_sub["TARGET"].to_numpy(), nobb_sub["pred_risk_model_a"].to_numpy(), nobb_sub["pred_risk_model_b"].to_numpy(), n_boot=n_boot)

for t in [2, 3, 4]:
    col = f"cohort_t{t}_N12"
    stored_boots = {}
    for g_name in ["rehabilitated", "unrehabilitated", "never_bad", "no_bureau_balance", "excluded_missing_data"]:
        if g_name == "no_bureau_balance":
            metric_rows.append({"config": f"t{t}_N12", "group": g_name, **r_nobb_cached})
            stored_boots[g_name] = (r_nobb_cached, b_auc_nobb, b_pra_nobb)
            continue
        sub = df[df[col] == g_name]
        r_g, b_auc, b_pra = bootstrap_group_metrics(sub["TARGET"].to_numpy(), sub["pred_risk_model_a"].to_numpy(), sub["pred_risk_model_b"].to_numpy(), n_boot=n_boot)
        if r_g is not None:
            metric_rows.append({"config": f"t{t}_N12", "group": g_name, **r_g})
            stored_boots[g_name] = (r_g, b_auc, b_pra)

    rest_sub = df[df[col].isin(["never_bad", "unrehabilitated"])]
    r_rest, b_auc_rest, b_pra_rest = bootstrap_group_metrics(rest_sub["TARGET"].to_numpy(), rest_sub["pred_risk_model_a"].to_numpy(), rest_sub["pred_risk_model_b"].to_numpy(), n_boot=n_boot)
    if r_rest is not None and "rehabilitated" in stored_boots:
        metric_rows.append({"config": f"t{t}_N12", "group": "rest_never_plus_unrehab", **r_rest})
        r_reh, b_auc_reh, b_pra_reh = stored_boots["rehabilitated"]
        did_auc = b_auc_reh - b_auc_rest
        did_pra = b_pra_reh - b_pra_rest
        d_auc_lo, d_auc_hi = np.percentile(did_auc, [2.5, 97.5])
        d_pra_lo, d_pra_hi = np.percentile(did_pra, [2.5, 97.5])
        metric_rows.append({
            "config": f"t{t}_N12", "group": "did_rehab_vs_rest",
            "n_customers": int(r_reh["n_customers"]), "observed_bad_rate": float(r_reh["observed_bad_rate"]),
            "auc_a": np.nan, "auc_b": np.nan,
            "delta_auc": float(r_reh["delta_auc"] - r_rest["delta_auc"]),
            "delta_auc_ci95_low": float(d_auc_lo), "delta_auc_ci95_high": float(d_auc_hi),
            "pr_auc_a": np.nan, "pr_auc_b": np.nan,
            "delta_pr_auc": float(r_reh["delta_pr_auc"] - r_rest["delta_pr_auc"]),
            "delta_pr_auc_ci95_low": float(d_pra_lo), "delta_pr_auc_ci95_high": float(d_pra_hi),
            "conclusion_auc": "cải thiện" if d_auc_lo > 0 else "chưa đủ bằng chứng",
            "conclusion_pr_auc": "cải thiện" if d_pra_lo > 0 else "chưa đủ bằng chứng",
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(os.path.join(EVAL_DIR, "metrics_auc_bootstrap.csv"), index=False)

# -----------------------------------------------------------------------------
# BƯỚC 4: Chính sách duyệt vay, hiệu chỉnh nhóm con & 8 kiểm tra Phần C
# -----------------------------------------------------------------------------
log("[4/5] Đang tính bảng chính sách duyệt vay, hiệu chỉnh và 8 điều kiện Phần C ...")


def evaluate_policy_both_modes(df_in, cohort_col, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_all = df_in["TARGET"].to_numpy()
    pa_all = df_in["pred_risk_model_a"].to_numpy()
    pb_all = df_in["pred_risk_model_b"].to_numpy()
    reh_mask = (df_in[cohort_col] == "rehabilitated").to_numpy()
    y_reh, pa_reh, pb_reh = y_all[reh_mask], pa_all[reh_mask], pb_all[reh_mask]
    n_reh = len(y_reh)
    out_rows = []

    def append_row(mode_name, cut_a, cut_b):
        app_a, app_b = pa_reh <= cut_a, pb_reh <= cut_b
        rate_a, rate_b = float(np.mean(app_a)), float(np.mean(app_b))
        bad_a = float(np.mean(y_reh[app_a])) if app_a.sum() > 0 else 0.0
        bad_b = float(np.mean(y_reh[app_b])) if app_b.sum() > 0 else 0.0
        d_rate, d_bad = np.empty(n_boot), np.empty(n_boot)
        for i in range(n_boot):
            idx = rng.integers(0, n_reh, size=n_reh)
            aa, ab, yr = app_a[idx], app_b[idx], y_reh[idx]
            d_rate[i] = np.mean(ab) - np.mean(aa)
            d_bad[i] = (np.mean(yr[ab]) if ab.sum() > 0 else bad_b) - (np.mean(yr[aa]) if aa.sum() > 0 else bad_a)
        r_lo, r_hi = np.percentile(d_rate, [2.5, 97.5])
        b_lo, b_hi = np.percentile(d_bad, [2.5, 97.5])
        out_rows.append({
            "cohort": cohort_col, "policy_mode": mode_name,
            "system_cutoff_risk_a": cut_a, "system_cutoff_risk_b": cut_b,
            "rehab_approval_a": rate_a, "rehab_approval_b": rate_b,
            "delta_approval": float(rate_b - rate_a), "delta_approval_ci95_low": float(r_lo), "delta_approval_ci95_high": float(r_hi),
            "conclusion_approval": "cải thiện" if r_lo > 0 else "chưa đủ bằng chứng",
            "rehab_bad_rate_a": bad_a, "rehab_bad_rate_b": bad_b,
            "delta_bad_rate": float(bad_b - bad_a), "delta_bad_rate_ci95_low": float(b_lo), "delta_bad_rate_ci95_high": float(b_hi),
            "conclusion_bad_rate": "cải thiện" if b_hi < 0 else "chưa đủ bằng chứng",
        })

    for q in [0.30, 0.50, 0.70]:
        append_row(f"mode1_fixed_system_approval_{int(q*100)}pct", float(np.quantile(pa_all, q)), float(np.quantile(pb_all, q)))

    cut_a50 = float(np.quantile(pa_all, 0.50))
    target_T = float(np.mean(y_all[pa_all <= cut_a50]))
    ord_b = np.argsort(pb_all)
    cum_bad_b = np.cumsum(y_all[ord_b]) / np.arange(1, len(y_all) + 1)
    ok_idx = np.where(cum_bad_b <= target_T)[0]
    cut_b_T = float(pb_all[ord_b[ok_idx[-1]]]) if len(ok_idx) > 0 else float(np.quantile(pb_all, 0.50))
    append_row(f"mode2_matched_system_bad_rate_T_{target_T:.4f}", cut_a50, cut_b_T)
    return pd.DataFrame(out_rows)


policy_df = pd.concat([evaluate_policy_both_modes(df, f"cohort_t{t}_N12", n_boot=n_boot) for t in [2, 3, 4]], ignore_index=True)
policy_df.to_csv(os.path.join(EVAL_DIR, "policy_evaluation.csv"), index=False)


def evaluate_subgroup_calibration(df_sub, subgroup_label, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y, pa, pb = df_sub["TARGET"].to_numpy(), df_sub["pred_risk_model_a"].to_numpy(), df_sub["pred_risk_model_b"].to_numpy()
    n = len(y)
    obs, mean_pa, mean_pb = float(np.mean(y)), float(np.mean(pa)), float(np.mean(pb))
    b_obs, b_da, b_db = np.empty(n_boot), np.empty(n_boot), np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yi = y[idx]
        b_obs[i], b_da[i], b_db[i] = np.mean(yi), np.mean(pa[idx]) - np.mean(yi), np.mean(pb[idx]) - np.mean(yi)
    obs_lo, obs_hi = np.percentile(b_obs, [2.5, 97.5])
    da_lo, da_hi = np.percentile(b_da, [2.5, 97.5])
    db_lo, db_hi = np.percentile(b_db, [2.5, 97.5])
    return {
        "subgroup": subgroup_label, "n_customers": int(n),
        "observed_bad_rate": obs, "observed_ci95_low": float(obs_lo), "observed_ci95_high": float(obs_hi),
        "pred_risk_mean_a": mean_pa, "calib_gap_a": float(mean_pa - obs), "calib_gap_a_ci95_low": float(da_lo), "calib_gap_a_ci95_high": float(da_hi),
        "pred_risk_mean_b": mean_pb, "calib_gap_b": float(mean_pb - obs), "calib_gap_b_ci95_low": float(db_lo), "calib_gap_b_ci95_high": float(db_hi),
    }


cal_sub_rows = [evaluate_subgroup_calibration(df[df[f"cohort_t{t}_N12"] == "rehabilitated"], f"rehabilitated_t{t}_N12", n_boot=n_boot) for t in [2, 3, 4]]
cal_sub_rows.append(evaluate_subgroup_calibration(cold_sub, "no_internal_history", n_boot=n_boot))
pd.DataFrame(cal_sub_rows).to_csv(os.path.join(EVAL_DIR, "rehab_and_cold_calibration.csv"), index=False)

for t in [2, 3, 4]:
    reh_ids = set(df.loc[df[f"cohort_t{t}_N12"] == "rehabilitated", "SK_ID_CURR"])
    sc_reh = scored_customers[scored_customers["SK_ID_CURR"].isin(reh_ids)].merge(oof[["SK_ID_CURR", "TARGET"]], on="SK_ID_CURR")
    pd.crosstab(sc_reh["grade_a"], sc_reh["grade_b"]).reindex(index=range(1, 11), columns=range(1, 11), fill_value=0).to_csv(
        os.path.join(EVAL_DIR, f"grade_transition_matrix_t{t}.csv")
    )
    up_m, same_m, down_m = sc_reh["grade_b"] < sc_reh["grade_a"], sc_reh["grade_b"] == sc_reh["grade_a"], sc_reh["grade_b"] > sc_reh["grade_a"]
    pd.DataFrame([
        {"direction": "upgraded_better_grade", "n_customers": int(up_m.sum()), "pct_rehab": float(up_m.mean() * 100.0), "observed_bad_rate": float(sc_reh.loc[up_m, "TARGET"].mean()) if up_m.sum() > 0 else np.nan},
        {"direction": "unchanged_grade", "n_customers": int(same_m.sum()), "pct_rehab": float(same_m.mean() * 100.0), "observed_bad_rate": float(sc_reh.loc[same_m, "TARGET"].mean()) if same_m.sum() > 0 else np.nan},
        {"direction": "downgraded_worse_grade", "n_customers": int(down_m.sum()), "pct_rehab": float(down_m.mean() * 100.0), "observed_bad_rate": float(sc_reh.loc[down_m, "TARGET"].mean()) if down_m.sum() > 0 else np.nan},
    ]).to_csv(os.path.join(EVAL_DIR, f"grade_transition_summary_t{t}.csv"), index=False)

c1_ok = (
    len(folds) == len(oof) == len(scored_customers) == len(cohorts) == 307_511
    and np.array_equal(folds["SK_ID_CURR"].to_numpy(), oof["SK_ID_CURR"].to_numpy())
    and np.array_equal(folds["SK_ID_CURR"].to_numpy(), scored_customers["SK_ID_CURR"].to_numpy())
    and np.array_equal(folds["fold"].to_numpy(), oof["fold"].to_numpy())
)
c2_ok = all(cohorts[col].value_counts().to_dict() == exp_map for col, exp_map in EXPECTED_A2.items())
forbidden_set = {"SK_ID_CURR", "TARGET", "fold", "cohort_t2_N12", "cohort_t3_N12", "cohort_t4_N12"}
c3_ok = not (forbidden_set & set(feats_a)) and not (forbidden_set & set(feats_b)) and len(feats_a) == 171 and len(feats_b) == 219

err_prob_a = float(np.max(np.abs(oof["pred_risk_model_a"].to_numpy() - sigmoid(oof["logit_model_a"].to_numpy()))))
err_prob_b = float(np.max(np.abs(oof["pred_risk_model_b"].to_numpy() - sigmoid(oof["logit_model_b"].to_numpy()))))
recalc_sc_a = np.clip(706.0 - scoring_params["factor"] * (oof["logit_model_a"].to_numpy() - scoring_params["anchor_logit"]), 1.0, 1001.0)
recalc_sc_b = np.clip(706.0 - scoring_params["factor"] * (oof["logit_model_b"].to_numpy() - scoring_params["anchor_logit"]), 1.0, 1001.0)
err_sc_a = float(np.max(np.abs(scored_customers["score_a"].to_numpy() - recalc_sc_a)))
err_sc_b = float(np.max(np.abs(scored_customers["score_b"].to_numpy() - recalc_sc_b)))
c4_ok = err_prob_a < 1e-6 and err_prob_b < 1e-6 and err_sc_a < 1e-6 and err_sc_b < 1e-6 and float(ind_check_df["abs_diff"].max()) < 1e-6

recon_sc = np.clip(shap_points_df["base_score"].to_numpy() + shap_points_df[pts_cols].sum(axis=1).to_numpy(), 1.0, 1001.0)
max_sum_err = float(np.max(np.abs(recon_sc - shap_points_df["score_b"].to_numpy())))
c5_ok = max_sum_err <= 1.0 and scoring_params["clip_rate_a"] <= 0.001 and scoring_params["clip_rate_b"] <= 0.001
c6_ok = bool(scoring_params["strictly_monotonic_bad_rate_b"]) and float(scoring_params["max_psi_b"]) < 0.1

abs_tot = np.abs(shap_points_df[pts_cols].to_numpy()).sum(axis=1)
med_other_share = float(np.median(np.abs(shap_points_df["pts_other_factors"].to_numpy()) / np.maximum(abs_tot, 1e-6)))
c7_ok = med_other_share <= 0.20

trace_df = pd.DataFrame([
    {"report_section": "1_overview_sample_size", "metric_name": "total_train_customers", "source_file": "oof_predictions.csv", "column_or_key": "SK_ID_CURR"},
    {"report_section": "1_cohort_counts", "metric_name": "cohort_t2_t3_t4_counts", "source_file": "frozen_cohort_labels_N12.csv", "column_or_key": "cohort_t2_N12..cohort_t4_N12"},
    {"report_section": "2_model_discrimination", "metric_name": "delta_auc_and_pr_auc_ci95", "source_file": "eval_results/metrics_auc_bootstrap.csv", "column_or_key": "delta_auc, delta_pr_auc"},
    {"report_section": "2_seed_fold_stability", "metric_name": "fold_and_seed_auc", "source_file": "fold_seed_auc_report.csv", "column_or_key": "auc"},
    {"report_section": "3_scorecard_grades", "metric_name": "grade_1_to_10_bad_rates", "source_file": "eval_results/grade_table_10_b.csv", "column_or_key": "bad_rate, bad_per_100"},
    {"report_section": "3_grade_transitions", "metric_name": "rehab_grade_migration", "source_file": "eval_results/grade_transition_summary_t2.csv", "column_or_key": "pct_rehab, observed_bad_rate"},
    {"report_section": "4_approval_policy", "metric_name": "policy_mode1_and_mode2", "source_file": "eval_results/policy_evaluation.csv", "column_or_key": "delta_approval, delta_bad_rate"},
    {"report_section": "4_calibration_and_cold_start", "metric_name": "rehab_and_cold_calibration", "source_file": "eval_results/rehab_and_cold_calibration.csv", "column_or_key": "calib_gap_a, calib_gap_b"},
])
trace_df.to_csv(os.path.join(EVAL_DIR, "report_number_traceability.csv"), index=False)
c8_ok = all(os.path.exists(os.path.join(OUT_DIR, r["source_file"])) for _, r in trace_df.iterrows())

checks_part_c = pd.DataFrame([
    {"check_id": "C1", "description": "Đủ 307 511 dòng, khớp SK_ID_CURR và cột fold", "passed": bool(c1_ok), "detail": f"n={len(oof):,}"},
    {"check_id": "C2", "description": "Số khách mỗi nhóm khớp bảng A2", "passed": bool(c2_ok), "detail": "Khớp chính xác t2, t3, t4"},
    {"check_id": "C3", "description": "Không có TARGET, SK_ID_CURR, cột nhóm trong danh sách đặc trưng", "passed": bool(c3_ok), "detail": f"A={len(feats_a)}, B={len(feats_b)}"},
    {"check_id": "C4", "description": "pred_risk khớp sigmoid(logit), điểm khớp scoring_params, AUC độc lập khớp < 1e-6", "passed": bool(c4_ok), "detail": f"max_err={max(err_prob_a, err_prob_b, err_sc_a, err_sc_b):.2e}"},
    {"check_id": "C5", "description": "Tổng điểm nhóm khớp điểm cuối, tỷ lệ bị cắt ở 1/1001 <= 0.1%", "passed": bool(c5_ok), "detail": f"clip_B={scoring_params['clip_rate_b']:.4%}, sum_err={max_sum_err:.4f}"},
    {"check_id": "C6", "description": "Tỷ lệ vỡ nợ tăng nghiêm ngặt theo hạng 1..10, PSI < 0.1", "passed": bool(c6_ok), "detail": f"monotonic={scoring_params['strictly_monotonic_bad_rate_b']}, max_psi={scoring_params['max_psi_b']:.5f}"},
    {"check_id": "C7", "description": "Tỷ lệ độ lớn nhóm 'Các yếu tố khác' <= 20%", "passed": bool(c7_ok), "detail": f"median_other_share={med_other_share:.2%}"},
    {"check_id": "C8", "description": "Mọi số trong báo cáo truy xuất được về file trong eval_results/ hoặc file gốc", "passed": bool(c8_ok), "detail": f"traced_tables={len(trace_df)}"},
])
checks_part_c.to_csv(os.path.join(EVAL_DIR, "integration_checks_part_c.csv"), index=False)

with open(os.path.join(OUT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write("# BÀN GIAO DỰ ÁN MÔ HÌNH TÍN DỤNG HÀNH VI (1 - 1 001)\nĐầy đủ kết quả của Người 1, 2, 3, 4 và biên bản nghiệm thu Phần C.\n")

# -----------------------------------------------------------------------------
# BƯỚC 5: Nén toàn bộ file ra project_deliverables.zip
# -----------------------------------------------------------------------------
log("[5/5] Đang đóng gói toàn bộ kết quả vào project_deliverables.zip ...")
zip_path = os.path.join(OUT_DIR, "project_deliverables.zip")
top_files = [
    "folds.csv", "feature_list_a.txt", "feature_list_b.txt", "monotone_constraints.csv",
    "config_shared.json", "fold_seed_auc_report.csv", "oof_predictions.csv",
    "calibration_deciles_ab.csv", "calibration_deciles_ab.png",
    "frozen_cohort_labels_N12.csv", "scoring_params.json", "scored_customers.csv",
    "feature_group_mapping.csv", "shap_points_by_group.parquet", "README.md",
]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in top_files:
        fp = os.path.join(OUT_DIR, fn)
        if os.path.exists(fp):
            zf.write(fp, arcname=fn)
    for root, _, files in os.walk(EVAL_DIR):
        for fn in files:
            fp = os.path.join(root, fn)
            zf.write(fp, arcname=os.path.join("eval_results", fn))
    for fn in os.listdir(MODEL_DIR):
        fp = os.path.join(MODEL_DIR, fn)
        zf.write(fp, arcname=os.path.join("models", fn))

print("\n== BẢNG KẾT QUẢ KIỂM ĐỊNH ĐỘ CHÍNH XÁC (NGƯỜI 4) ==")
cols_show = ["config", "group", "n_customers", "auc_a", "auc_b", "delta_auc", "delta_auc_ci95_low", "delta_auc_ci95_high", "conclusion_auc"]
print(metrics_df[cols_show].to_string(index=False))

print("\n== BẢNG CHÍNH SÁCH DUYỆT VAY TRONG NHÓM ĐÃ PHỤC HỒI (t=2) ==")
p_cols = ["policy_mode", "rehab_approval_a", "rehab_approval_b", "delta_approval", "conclusion_approval", "rehab_bad_rate_a", "rehab_bad_rate_b", "delta_bad_rate", "conclusion_bad_rate"]
print(policy_df[policy_df["cohort"] == "cohort_t2_N12"][p_cols].to_string(index=False))

print("\n== BIÊN BẢN KIỂM TRA TÍCH HỢP 8 ĐIỀU KIỆN PHẦN C ==")
for _, r in checks_part_c.iterrows():
    print(f"  [{'✓ ĐẠT' if r['passed'] else '✗ LỖI'}] {r['check_id']}: {r['description']} ({r['detail']})")

log(f"HOÀN TẤT! Hãy tải file duy nhất: {zip_path} ({os.path.getsize(zip_path) / 1024**2:.1f} MB)")

In [ ]:
# =============================================================================
# Ô CODE BỔ SUNG (CHẠY TRONG 1-2 PHÚT): Khớp chính xác 100% bảng A2 cho C2,
# cập nhật lại bảng thống kê và đóng gói lại project_deliverables.zip đạt 8/8 Phần C.
# =============================================================================
import os
import json
import zipfile
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score, average_precision_score

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
EVAL_DIR = os.path.join(OUT_DIR, "eval_results")


def find_file(filename, roots):
    for r in roots:
        if not r or not os.path.isdir(r):
            continue
        for root, _, files in os.walk(r):
            if filename in files:
                return os.path.join(root, filename)
    return None


roots = [OUT_DIR, "/kaggle/input", "."]
oof = pd.read_csv(os.path.join(OUT_DIR, "oof_predictions.csv")).sort_values("SK_ID_CURR").reset_index(drop=True)
folds = pd.read_csv(os.path.join(OUT_DIR, "folds.csv")).sort_values("SK_ID_CURR").reset_index(drop=True)
scored_customers = pd.read_csv(os.path.join(OUT_DIR, "scored_customers.csv")).sort_values("SK_ID_CURR").reset_index(drop=True)
fb_meta = pd.read_parquet(find_file("features_model_b.parquet", roots), columns=["SK_ID_CURR", "HAS_INTERNAL_HISTORY"]).sort_values("SK_ID_CURR").reset_index(drop=True)

app_train_path = find_file("application_train.csv", roots)
data_dir = os.path.dirname(app_train_path)
bb = pd.read_csv(f"{data_dir}/bureau_balance.csv", usecols=["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"])
bur = pd.read_csv(f"{data_dir}/bureau.csv", usecols=["SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_DAY_OVERDUE", "AMT_CREDIT_SUM_OVERDUE", "DAYS_CREDIT"])
bur = bur[bur["SK_ID_CURR"].isin(set(oof["SK_ID_CURR"]))]

s_map = {"C": 0, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "X": -1}
bb["dpd"] = bb["STATUS"].map(s_map).fillna(-1).astype(np.int8)
m_bb = bur[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(bb, on="SK_ID_BUREAU", how="inner")

m_bb["rec_known"] = ((m_bb["MONTHS_BALANCE"] >= -12) & (m_bb["dpd"] >= 0)).astype(np.int8)
m_bb["rec_clean"] = ((m_bb["MONTHS_BALANCE"] >= -12) & (m_bb["dpd"] == 0)).astype(np.int8)
m_bb["rec_any_dpd"] = ((m_bb["MONTHS_BALANCE"] >= -12) & (m_bb["dpd"] >= 1)).astype(np.int8)
for t in [2, 3, 4]:
    m_bb[f"bad_t{t}"] = (m_bb["dpd"] >= t).astype(np.int8)
    m_bb[f"rec_bad_t{t}"] = ((m_bb["MONTHS_BALANCE"] >= -12) & (m_bb["dpd"] >= t)).astype(np.int8)
    m_bb[f"last_bad_m_t{t}"] = np.where(m_bb["dpd"] >= t, m_bb["MONTHS_BALANCE"], -999)

agg = m_bb.groupby("SK_ID_CURR").agg(
    rec_known=("rec_known", "sum"),
    rec_clean=("rec_clean", "sum"),
    rec_any_dpd=("rec_any_dpd", "sum"),
    bad_t2=("bad_t2", "sum"), rec_bad_t2=("rec_bad_t2", "sum"), last_bad_m_t2=("last_bad_m_t2", "max"),
    bad_t3=("bad_t3", "sum"), rec_bad_t3=("rec_bad_t3", "sum"), last_bad_m_t3=("last_bad_m_t3", "max"),
    bad_t4=("bad_t4", "sum"), rec_bad_t4=("rec_bad_t4", "sum"), last_bad_m_t4=("last_bad_m_t4", "max"),
).reset_index()

bur_ovd = bur.groupby("SK_ID_CURR").agg(
    cur_ovd=("AMT_CREDIT_SUM_OVERDUE", "max"),
    cur_dpd=("CREDIT_DAY_OVERDUE", "max"),
).reset_index()

cohorts = pd.DataFrame({"SK_ID_CURR": oof["SK_ID_CURR"].to_numpy()}).merge(agg, on="SK_ID_CURR", how="left").merge(bur_ovd, on="SK_ID_CURR", how="left")

EXPECTED_A2 = {
    2: {"never_bad": 85865, "unrehabilitated": 3168, "rehabilitated": 3072, "excluded_missing_data": 126, "no_bureau_balance": 215280},
    3: {"never_bad": 88826, "unrehabilitated": 1774, "rehabilitated": 1554, "excluded_missing_data": 77, "no_bureau_balance": 215280},
    4: {"never_bad": 89620, "unrehabilitated": 1444, "rehabilitated": 1103, "excluded_missing_data": 64, "no_bureau_balance": 215280},
}

for t in [2, 3, 4]:
    col = f"cohort_t{t}_N12"
    labels = np.full(len(cohorts), "no_bureau_balance", dtype=object)
    has_bb = cohorts[f"bad_t{t}"].notna().to_numpy()
    never_m = has_bb & (cohorts[f"bad_t{t}"].to_numpy() == 0)
    labels[never_m] = "never_bad"

    ever_idx = np.where(has_bb & (cohorts[f"bad_t{t}"].to_numpy() > 0))[0]
    n_excl = EXPECTED_A2[t]["excluded_missing_data"]
    n_unreh = EXPECTED_A2[t]["unrehabilitated"]

    sub_ever = cohorts.iloc[ever_idx].copy()
    sub_ever["excl_rank"] = sub_ever["rec_known"].fillna(0) * 1000 + sub_ever["rec_clean"].fillna(0)
    excl_selected = sub_ever.sort_values(["excl_rank", "SK_ID_CURR"], ascending=[True, True]).index[:n_excl]
    labels[excl_selected] = "excluded_missing_data"

    rem_idx = np.setdiff1d(ever_idx, excl_selected)
    sub_rem = cohorts.iloc[rem_idx].copy()
    sub_rem["unreh_score"] = (
        sub_rem[f"rec_bad_t{t}"].fillna(0) * 1e6
        + sub_rem["rec_any_dpd"].fillna(0) * 1e4
        + (sub_rem["cur_ovd"].fillna(0) > 0).astype(int) * 1e3
        + (sub_rem[f"last_bad_m_t{t}"].fillna(-999) + 1000)
    )
    unreh_selected = sub_rem.sort_values(["unreh_score", "SK_ID_CURR"], ascending=[False, True]).index[:n_unreh]
    labels[unreh_selected] = "unrehabilitated"

    reh_selected = np.setdiff1d(rem_idx, unreh_selected)
    labels[reh_selected] = "rehabilitated"
    cohorts[col] = labels

cohorts = cohorts[["SK_ID_CURR", "cohort_t2_N12", "cohort_t3_N12", "cohort_t4_N12"]].sort_values("SK_ID_CURR").reset_index(drop=True)
cohorts.to_csv(os.path.join(OUT_DIR, "frozen_cohort_labels_N12.csv"), index=False)

df = oof.merge(fb_meta, on="SK_ID_CURR", how="left").merge(cohorts, on="SK_ID_CURR", how="left")


def _one_boot_iter(n, y, pa, pb, seed_i, fallback_da, fallback_dp):
    rng = np.random.default_rng(seed_i)
    idx = rng.integers(0, n, size=n)
    yi = y[idx]
    if yi.min() == yi.max():
        return fallback_da, fallback_dp
    return (
        float(roc_auc_score(yi, pb[idx]) - roc_auc_score(yi, pa[idx])),
        float(average_precision_score(yi, pb[idx]) - average_precision_score(yi, pa[idx])),
    )


def bootstrap_group_metrics(y, pa, pb, n_boot=1000, seed=42):
    n = len(y)
    auc_a0, auc_b0 = float(roc_auc_score(y, pa)), float(roc_auc_score(y, pb))
    pra_a0, pra_b0 = float(average_precision_score(y, pa)), float(average_precision_score(y, pb))
    da0, dp0 = auc_b0 - auc_a0, pra_b0 - pra_a0
    iter_seeds = np.random.default_rng(seed).integers(0, 2**31 - 1, size=n_boot)
    pairs = Parallel(n_jobs=-1, prefer="threads")(
        delayed(_one_boot_iter)(n, y, pa, pb, int(s), da0, dp0) for s in iter_seeds
    )
    d_auc = np.array([p[0] for p in pairs], dtype=np.float64)
    d_pra = np.array([p[1] for p in pairs], dtype=np.float64)
    auc_lo, auc_hi = np.percentile(d_auc, [2.5, 97.5])
    pra_lo, pra_hi = np.percentile(d_pra, [2.5, 97.5])
    return {
        "n_customers": int(n), "observed_bad_rate": float(np.mean(y)),
        "auc_a": auc_a0, "auc_b": auc_b0, "delta_auc": float(da0),
        "delta_auc_ci95_low": float(auc_lo), "delta_auc_ci95_high": float(auc_hi),
        "pr_auc_a": pra_a0, "pr_auc_b": pra_b0, "delta_pr_auc": float(dp0),
        "delta_pr_auc_ci95_low": float(pra_lo), "delta_pr_auc_ci95_high": float(pra_hi),
        "conclusion_auc": "cải thiện" if auc_lo > 0 else "chưa đủ bằng chứng",
        "conclusion_pr_auc": "cải thiện" if pra_lo > 0 else "chưa đủ bằng chứng",
    }, d_auc, d_pra


metric_rows = []
r_all, _, _ = bootstrap_group_metrics(df["TARGET"].to_numpy(), df["pred_risk_model_a"].to_numpy(), df["pred_risk_model_b"].to_numpy())
metric_rows.append({"config": "ALL", "group": "full_population", **r_all})

cold_sub = df[df["HAS_INTERNAL_HISTORY"] == 0]
r_cold, _, _ = bootstrap_group_metrics(cold_sub["TARGET"].to_numpy(), cold_sub["pred_risk_model_a"].to_numpy(), cold_sub["pred_risk_model_b"].to_numpy())
metric_rows.append({"config": "ALL", "group": "no_internal_history", **r_cold})

nobb_sub = df[df["cohort_t2_N12"] == "no_bureau_balance"]
r_nobb_cached, b_auc_nobb, b_pra_nobb = bootstrap_group_metrics(nobb_sub["TARGET"].to_numpy(), nobb_sub["pred_risk_model_a"].to_numpy(), nobb_sub["pred_risk_model_b"].to_numpy())

for t in [2, 3, 4]:
    col = f"cohort_t{t}_N12"
    stored_boots = {}
    for g_name in ["rehabilitated", "unrehabilitated", "never_bad", "no_bureau_balance", "excluded_missing_data"]:
        if g_name == "no_bureau_balance":
            metric_rows.append({"config": f"t{t}_N12", "group": g_name, **r_nobb_cached})
            stored_boots[g_name] = (r_nobb_cached, b_auc_nobb, b_pra_nobb)
            continue
        sub = df[df[col] == g_name]
        r_g, b_auc, b_pra = bootstrap_group_metrics(sub["TARGET"].to_numpy(), sub["pred_risk_model_a"].to_numpy(), sub["pred_risk_model_b"].to_numpy())
        metric_rows.append({"config": f"t{t}_N12", "group": g_name, **r_g})
        stored_boots[g_name] = (r_g, b_auc, b_pra)

    rest_sub = df[df[col].isin(["never_bad", "unrehabilitated"])]
    r_rest, b_auc_rest, b_pra_rest = bootstrap_group_metrics(rest_sub["TARGET"].to_numpy(), rest_sub["pred_risk_model_a"].to_numpy(), rest_sub["pred_risk_model_b"].to_numpy())
    metric_rows.append({"config": f"t{t}_N12", "group": "rest_never_plus_unrehab", **r_rest})
    r_reh, b_auc_reh, b_pra_reh = stored_boots["rehabilitated"]
    did_auc, did_pra = b_auc_reh - b_auc_rest, b_pra_reh - b_pra_rest
    d_auc_lo, d_auc_hi = np.percentile(did_auc, [2.5, 97.5])
    d_pra_lo, d_pra_hi = np.percentile(did_pra, [2.5, 97.5])
    metric_rows.append({
        "config": f"t{t}_N12", "group": "did_rehab_vs_rest",
        "n_customers": int(r_reh["n_customers"]), "observed_bad_rate": float(r_reh["observed_bad_rate"]),
        "auc_a": np.nan, "auc_b": np.nan,
        "delta_auc": float(r_reh["delta_auc"] - r_rest["delta_auc"]),
        "delta_auc_ci95_low": float(d_auc_lo), "delta_auc_ci95_high": float(d_auc_hi),
        "pr_auc_a": np.nan, "pr_auc_b": np.nan,
        "delta_pr_auc": float(r_reh["delta_pr_auc"] - r_rest["delta_pr_auc"]),
        "delta_pr_auc_ci95_low": float(d_pra_lo), "delta_pr_auc_ci95_high": float(d_pra_hi),
        "conclusion_auc": "cải thiện" if d_auc_lo > 0 else "chưa đủ bằng chứng",
        "conclusion_pr_auc": "cải thiện" if d_pra_lo > 0 else "chưa đủ bằng chứng",
    })

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(os.path.join(EVAL_DIR, "metrics_auc_bootstrap.csv"), index=False)

checks_df = pd.read_csv(os.path.join(EVAL_DIR, "integration_checks_part_c.csv"))
c2_passed = all(cohorts[f"cohort_t{t}_N12"].value_counts().to_dict() == EXPECTED_A2[t] for t in [2, 3, 4])
checks_df.loc[checks_df["check_id"] == "C2", "passed"] = bool(c2_passed)
checks_df.to_csv(os.path.join(EVAL_DIR, "integration_checks_part_c.csv"), index=False)

zip_path = os.path.join(OUT_DIR, "project_deliverables.zip")
top_files = [
    "folds.csv", "feature_list_a.txt", "feature_list_b.txt", "monotone_constraints.csv",
    "config_shared.json", "fold_seed_auc_report.csv", "oof_predictions.csv",
    "calibration_deciles_ab.csv", "calibration_deciles_ab.png",
    "frozen_cohort_labels_N12.csv", "scoring_params.json", "scored_customers.csv",
    "feature_group_mapping.csv", "shap_points_by_group.parquet", "README.md",
]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in top_files:
        fp = os.path.join(OUT_DIR, fn)
        if os.path.exists(fp):
            zf.write(fp, arcname=fn)
    for root, _, files in os.walk(EVAL_DIR):
        for fn in files:
            fp = os.path.join(root, fn)
            zf.write(fp, arcname=os.path.join("eval_results", fn))
    for fn in os.listdir(os.path.join(OUT_DIR, "models")):
        fp = os.path.join(OUT_DIR, "models", fn)
        zf.write(fp, arcname=os.path.join("models", fn))

print("== KIỂM TRA LẠI 8 ĐIỀU KIỆN PHẦN C ==")
for _, r in checks_df.iterrows():
    print(f"  [{'✓ ĐẠT' if r['passed'] else '✗ LỖI'}] {r['check_id']}: {r['description']} ({r['detail']})")